In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from pyspark.sql.window import Window

In [0]:
%sql
select * from fintech360.bronze.loans;

In [0]:
df = spark.read.table("fintech360.bronze.loans")
display(df)

# Validate expected columns and data types

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DateType, DecimalType, IntegerType, LongType
)

LOAN_PARENT_SCHEMA = StructType([
    StructField("loan_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("application_id", StringType(), True),
    StructField("loan_source", StringType(), True),
    StructField("loan_purpose", StringType(), True),
    StructField("principal_amount", DecimalType(18, 2), True),
    StructField("annual_interest_rate_pct", DecimalType(9, 6), True),
    StructField("tenure_months", IntegerType(), True),
    StructField("loan_status", StringType(), True),
    StructField("application_ts", TimestampType(), True),
    StructField("disbursement_ts", TimestampType(), True),
    StructField("maturity_date", DateType(), True),
    StructField("currency", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("created_at", LongType(), True),
    StructField("updated_at", LongType(), True),
    StructField("loan_details_json", StringType(), True),
    StructField("_rescued_data", StringType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True)
])

In [0]:
from pyspark.sql import functions as F

def normalize_loan_schema(df, parent_schema=LOAN_PARENT_SCHEMA):
    parent_names = [field.name for field in parent_schema.fields]
    parent_fields = {field.name: field for field in parent_schema.fields}

    # Capture unexpected source columns
    extra_cols = [c for c in df.columns if c not in parent_names]

    if extra_cols:
        extra_json = F.to_json(
            F.struct(*[F.col(c) for c in extra_cols])
        )

        if "_rescued_data" in df.columns:
            df = df.withColumn(
                "_rescued_data",
                F.when(
                    F.col("_rescued_data").isNull(),
                    extra_json
                ).otherwise(
                    F.concat_ws(
                        "\n",
                        F.col("_rescued_data"),
                        extra_json
                    )
                )
            )
        else:
            df = df.withColumn("_rescued_data", extra_json)

    # Normalize identifiers such as 1000.0 to 1000
    for col_name in ["loan_id", "customer_id"]:
        if col_name in df.columns:
            value = F.trim(F.col(col_name).cast("string"))
            numeric_value = value.cast("decimal(38,10)")

            df = df.withColumn(
                col_name,
                F.when(
                    value.rlike(r"^[+-]?\d+\.0+$"),
                    numeric_value.cast("decimal(38,0)").cast("string")
                ).otherwise(value)
            )

    # Add missing fields and cast to parent types
    for name, field in parent_fields.items():
        if name not in df.columns:
            df = df.withColumn(
                name,
                F.lit(None).cast(field.dataType)
            )
        else:
            df = df.withColumn(
                name,
                F.col(name).cast(field.dataType)
            )

    return df.select(*parent_names)

In [0]:
df = normalize_loan_schema(df)

df.printSchema()
display(df.limit(10))

In [0]:
(df.write
    .mode('append')
    .format('delta')
    .saveAsTable('fintech360.schema_validation.loans'))